# BÀI 3. MÃ ĐI TUẦN (MADITUAN.*)

## Mô tả bài toán:
Trong bàn cờ tổng quát kích thước N×N, con mã được đặt tại vị trí **x0, y0**. Hãy chỉ ra lộ trình con mã đi qua tất cả các ô mỗi ô chỉ di qua **một lần và trở lại vị trí ban đầu** (Hành trình khép kín).

**Giới hạn**: 2≤N≤10

## Dữ liệu:
**Nhập từ bàn phím:**
- 3 số nguyên: **N x0 y0**
  - **N**: Kích thước bàn cờ N×N
  - **(x0, y0)**: Tọa độ điểm xuất phát của con mã trên bàn cờ

## Dữ liệu ra:
**Xuất ra màn hình:**
- Dòng đầu tiên ghi số **0** hoặc **1** (0 là không có lộ trình, 1 là có lộ trình)
- Nếu có, **N dòng tiếp theo**, mỗi dòng có **N số cách nhau bởi một khoảng trắng** biểu diễn bàn cờ và lộ trình di chuyển của con mã. Nếu giá trị số k là k, nghĩa là con mã đi qua ô của bàn cờ bằng nước đi thứ k.

## Ví dụ:

**Bàn phím:** `6 1 1`  
**Màn hình:** (có lộ trình từ (1,1))
```
1
1 22 27 30 3 20
26 31 2 21 8 29
23 36 25 28 19 4
32 13 34 7 16 9
35 24 11 14 5 18
12 33 6 17 10 15
```

**Bàn phím:** `3 1 1`  
**Màn hình:** 0 *(không có lộ trình cho N=3)*

## Hỗ trợ hệ thống:

In [1]:
import sys

# Tăng giới hạn đệ quy để tránh lỗi khi bàn cờ lớn
sys.setrecursionlimit(2000)

## Thuật toán: Quay lui (Backtracking) + Heuristic Warnsdorff

### Ý tưởng:
1. **Backtracking**: Duyệt tất cả các hướng di chuyển của quân mã (8 hướng chữ L)
2. **Heuristic Warnsdorff**: Ưu tiên di chuyển đến những ô có **ít lối thoát nhất** (ít ô chưa thăm xung quanh).
   - Điều này giúp tránh bị kẹt ở các góc/cạnh của bàn cờ
3. **Hành trình khép kín**: Kiểm tra ô cuối có thể nhảy về ô xuất phát không
4. **Tối ưu toán học**: Bàn cờ có N×N ô phải chẵn, vì vậy N lẻ không có lời giải

In [9]:
def giai_ma_di_tuan(n, start_x, start_y):
    """
    Giải bài toán Mã đi tuần sử dụng Backtracking + Heuristic Warnsdorff.
    
    Args:
        n: Kích thước bàn cờ (N×N)
        start_x, start_y: Vị trí xuất phát (0-based)
    
    Returns:
        Bàn cờ nếu có lộ trình, None nếu không có
    """
    # [TỐI ƯU TOÁN HỌC]: 
    # Hành trình khép kín trên bàn cờ N×N chỉ tồn tại khi tổng số ô (N²) chẵn.
    # Nếu N lẻ, không có lời giải -> return None để tiết kiệm thời gian
    if n % 2 != 0:
        return None

    board = [[0] * n for _ in range(n)]
    
    # 8 hướng di chuyển của quân mã (chữ L)
    dx = [-2, -2, -1, -1,  1, 1,  2, 2]
    dy = [-1,  1, -2,  2, -2, 2, -1, 1]

    def hop_le(x, y):
        """Kiểm tra ô (x, y) có hợp lệ và chưa thăm không"""
        return 0 <= x < n and 0 <= y < n and board[x][y] == 0

    def dem_loi_thoat(x, y):
        """Đếm số ô chưa thăm xung quanh ô (x, y)"""
        count = 0
        for i in range(8):
            if hop_le(x + dx[i], y + dy[i]):
                count += 1
        return count

    # [HÀM ĐẶC BIỆT]: Quay lui + Heuristic Warnsdorff
    def Try(x, y, step):
        board[x][y] = step
        
        # Base case: Đã đi qua tất cả N×N ô
        if step == n * n:
            # Kiểm tra xem từ ô cuối có thể nhảy 1 bước về lại ô xuất phát không
            for i in range(8):
                if x + dx[i] == start_x and y + dy[i] == start_y:
                    return True
            board[x][y] = 0
            return False

        # Thu thập các bước đi tiếp theo có thể
        next_moves = []
        for i in range(8):
            nx, ny = x + dx[i], y + dy[i]
            if hop_le(nx, ny):
                loi_thoat = dem_loi_thoat(nx, ny)
                next_moves.append((loi_thoat, nx, ny))
        
        # [HEURISTIC WARNSDORFF]: Sắp xếp theo số lối thoát tăng dần
        # Ưu tiên đi vào những ô có ít lối thoát nhất (rìa/góc bàn cờ)
        # Điều này giúp tránh bị kẹt không có đường ra
        next_moves.sort(key=lambda val: val[0])

        # Đi sâu vào nhánh đệ quy
        for _, nx, ny in next_moves:
            if Try(nx, ny, step + 1):
                return True
        
        # Backtrack: Phục hồi trạng thái nếu sai đường
        board[x][y] = 0
        return False

    if Try(start_x, start_y, 1):
        return board
    return None

## Ví dụ 1: N=6, xuất phát từ (1, 1)

In [26]:
# Test với N = 6, xuất phát từ (1, 1) - tọa độ 1-based thành 0-based
n = 6
x0, y0 = 1 - 1, 1 - 1  # Chuyển từ 1-based sang 0-based

ket_qua = giai_ma_di_tuan(n, x0, y0)

if ket_qua is None:
    print(0)
else:
    print(1)
    for row in ket_qua:
        print(" ".join(map(str, row)))

1
1 10 21 24 3 12
22 25 2 11 20 31
9 36 23 32 13 4
26 33 8 17 30 19
7 16 35 28 5 14
34 27 6 15 18 29


## Ví dụ 2: N=3 (Không có lộ trình)

In [25]:
# Test với N = 3
n = 3
x0, y0 = 1 - 1, 1 - 1

ket_qua = giai_ma_di_tuan(n, x0, y0)

if ket_qua is None:
    print(0)
else:
    print(1)
    for row in ket_qua:
        print(" ".join(map(str, row)))

0


## Chương trình chính (nhập từ bàn phím):

In [24]:
# Nhập dữ liệu từ bàn phím
try:
    inp = input("Nhập N x0 y0: ").split()
    if not inp:
        print(0)
    else:
        n = int(inp[0])
        # Tọa độ đề bài cho là 1-based, chuyển về 0-based để xử lý mảng
        x0 = int(inp[1]) - 1
        y0 = int(inp[2]) - 1
        
        ket_qua = giai_ma_di_tuan(n, x0, y0)

        if ket_qua is None:
            print(0)
        else:
            print(1)
            for row in ket_qua:
                print(" ".join(map(str, row)))
except (EOFError, ValueError, IndexError):
    print(0)

1
3 40 5 22 19 26 29 24
6 21 2 41 30 23 18 27
47 4 39 20 59 28 25 32
38 7 48 1 42 31 58 17
11 46 37 60 49 64 33 56
8 51 10 43 36 57 16 63
45 12 53 50 61 14 55 34
52 9 44 13 54 35 62 15


## Giải thích chi tiết:

### 1. Tối ưu toán học:
- **Bàn cờ N×N có N² ô phải là số chẵn** để có hành trình khép kín
- Nếu N lẻ → N² lẻ → không có lời giải → return None ngay

### 2. 8 hướng di chuyển của quân mã:
```
(-2,-1)  (-2,+1)
   ↙       ↘
(-1,-2)   (-1,+2)
   ↓       ↑
(+1,-2)   (+1,+2)
   ↗       ↖
(+2,-1)  (+2,+1)
```

### 3. Heuristic Warnsdorff:
- **Ý tưởng**: Luôn đi đến ô có ít lối thoát nhất
- **Lý do**: Tránh bị kẹt ở các góc/cạnh (những ô chỉ có 1-2 lối thoát)
- **Ưu điểm**: Tăng tốc độ tìm lời giải đáng kể

### 4. Hành trình khép kín:
- Sau khi di chuyển N² lần, kiểm tra xem từ ô cuối có nhảy về ô xuất phát được không
- Nếu có → tìm thấy lời giải
- Nếu không → backtrack tiếp tục tìm

### 5. Độ phức tạp:
- **Thời gian**: O(N²) với Heuristic Warnsdorff (rất hiệu quả)
- **Không gian**: O(N²) cho bàn cờ